# Dělení dat: zadání rozšíření cvičení 08

Notebook obsahuje krátké spustitelné ukázky, zadání a nápovědy. Vlastní postup patří do prázdných kódových buněk pod jednotlivými kroky. Ukázky používají oddělená malá data a proměnné s předponou `demo_`. Úlohy A, B a C jsou samostatné: každá pracuje se svým datovým souborem. Všechny soubory jsou syntetické.

Notebook spusťte ze složky `08-Data-preprocessing` nebo z kořene repozitáře. Pracovní složka musí umožnit přístup k dodané složce `data`. Popis dat je v `data/README.md`; balíčky jsou uvedené v kořenovém `requirements.txt`.

## Příprava

Následující buňka načte potřebné nástroje pro ukázky i vlastní práci. Úlohy A, B a C jsou po těchto importech samostatné.

Cestu ke složce dat si při načítání první úlohy uložte do `DATA` jako objekt `Path`. Při spuštění ze složky notebooku jde o `data`; z kořene repozitáře o `08-Data-preprocessing/data`. `pd.read_csv` přijímá cestu k souboru. `display` zobrazuje tabulky, `np` označuje NumPy a `pd` pandas.

In [ ]:
from pathlib import Path  # Cesty k souborům pro vlastní úlohy.
from io import StringIO  # Malá ukázka CSV uložená v paměti.
import numpy as np  # Číselná pole a chybějící hodnoty.
import pandas as pd  # Tabulky, CSV a časové údaje.
from IPython.display import display  # Čitelný výpis tabulky.
from sklearn.model_selection import train_test_split, GroupShuffleSplit, TimeSeriesSplit  # Tři způsoby dělení dat.
from sklearn.tree import DecisionTreeClassifier  # Jednoduchý klasifikační model.
from sklearn.pipeline import Pipeline  # Společné učení úprav a modelu.
from sklearn.impute import SimpleImputer  # Doplnění chybějících vstupů z tréninku.
from sklearn.linear_model import Ridge  # Regresní model pro číselný cíl.
from sklearn.metrics import accuracy_score, mean_absolute_error  # Metriky klasifikace a regrese.

### Ukázka: načtení tabulky

Tři ukázkové řádky jsou uložené v paměti. Při čtení souboru přijímá `read_csv` místo `StringIO` cestu; například objekt `Path`. Ukázka nepoužívá data vlastních úloh.

In [ ]:
demo_csv = StringIO("category,value\nA,2\nB,5\nA,7\n")  # Obsah samostatného malého CSV.
demo_table = pd.read_csv(demo_csv)  # Stejné načítací API jako pro soubor.
display(demo_table)  # Zobrazení načtených sloupců a řádků.
demo_values = demo_table["value"].to_numpy()  # Převod jednoho sloupce na pole.
print(demo_values)  # Pole lze dále vybírat pomocí indexů.

## A · Kategorie a intervaly

**Cíl:** porovnat stratifikaci podle kategorie a podle intervalů číselné veličiny. Zatím se neučí model.

Načtěte `data/stratification-records.csv` do `a_table`. Obsahuje 120 řádků, sloupec `state` se štítky `normal`, `warning`, `fault` a číselný sloupec `load`. Neobsahuje chybějící hodnoty ani skupinovou či časovou závislost. Pořadí řádků nemá časový význam.

Vytvořte pole kategorií `a_state`, pole hodnot `a_load` a pole pozic řádků `a_indices`. Pro převod sloupce na pole slouží `.to_numpy()`, pozice mohou vzniknout pomocí `numpy.arange`. Prohlédněte četnosti kategorií. V obou následujících děleních použijte 25 % řádků pro test a `random_state=42`.

### Ukázka: rozdělení podle kategorií

Malý soubor má dvě kategorie. Dělení se provádí nad pozicemi řádků; stejné pozice potom vybírají příslušné štítky.

In [ ]:
demo_labels = np.array(["A"] * 8 + ["B"] * 4)  # Dvě kategorie s různými četnostmi.
demo_positions = np.arange(len(demo_labels))  # Pozice řádků, které se budou dělit.
demo_train, demo_test = train_test_split(demo_positions, test_size=0.5, random_state=0, stratify=demo_labels)  # Rozdělení se zachováním podílů kategorií.
print(demo_train, demo_test)  # Oddělené pozice tréninku a testu.
print(np.unique(demo_labels[demo_test], return_counts=True))  # Kategorie a jejich počty v ukázkovém testu.

### A1 · Kategorie

Pomocí `train_test_split` rozdělte `a_indices` do `a_train_idx` a `a_test_idx`. Stratifikace má zachovat podíly `a_state`. Vytvořte tabulku počtů každé kategorie v celých datech, tréninku a testu. Ověřte, že se části nepřekrývají a dohromady obsahují všechny řádky.

<details><summary>Nápověda</summary>

Velikost testu určuje `test_size`, reprodukovatelnost `random_state` a skupiny pro stratifikaci `stratify`. Do posledního argumentu patří jedno označení pro každý řádek. Návratové hodnoty mají pořadí trénink, test. Četnosti poskytuje `value_counts`; neprůnik lze posoudit pomocí množin indexů.

</details>

### Ukázka: kvantilové intervaly

Číselná veličina se pro účely stratifikace převede na štítky intervalů. Dva intervaly této malé ukázky obsahují stejný počet hodnot, ale mají různou šířku.

In [ ]:
demo_numbers = np.array([2, 3, 4, 6, 9, 12, 17, 30])  # Samostatná sada osmi různých hodnot.
demo_bins = pd.qcut(demo_numbers, q=2, labels=False)  # Štítky dvou kvantilových intervalů.
print(demo_bins)  # Každá původní hodnota má právě jeden štítek.
demo_bin_train, demo_bin_test = train_test_split(np.arange(len(demo_numbers)), test_size=0.5, random_state=0, stratify=demo_bins)  # Intervalové štítky určují stratifikaci.
print(np.unique(demo_bins[demo_bin_test], return_counts=True))  # Zastoupení intervalů v ukázkovém testu.

### A2 · Intervaly

Vytvořte `a_bins`: čtyři kvantilové intervaly veličiny `a_load`, označené celými čísly. Pomocí těchto štítků proveďte nové dělení stejných indexů do `a_interval_train_idx` a `a_interval_test_idx`. Vytvořte tabulku četností intervalů v celých datech, tréninku a testu. Zkontrolujte také podíly původních kategorií v tomto testu.

<details><summary>Nápověda</summary>

U `pandas.qcut` určuje počet intervalů argument `q`; `labels=False` vrátí číselná označení. Nová označení mohou být argumentem `stratify`. Testovací četnosti nemusí být přesně čtvrtinové, pokud požadovaný počet není celé číslo.

</details>

Tady jsou intervaly součástí návrhu výběru nezávislých vzorků. Pokud by byly příznakem modelu, jejich hranice by se odhadovaly jen z trénovacích dat.

**Výstup:** tabulky četností kategorií a intervalů.

**K zamyšlení:** Co se v každém dělení zachovalo? Proč kvantilová stratifikace sama nezaručuje podíly stavů? Jaký problém mohou mít společné kategorie „stav × interval“?

[Dokumentace dělení](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html), [kvantilové intervaly](https://pandas.pydata.org/docs/reference/api/pandas.qcut.html).

## B · Nová zařízení

**Cíl:** odhadnout kvalitu modelu pro zařízení, která nebyla v tréninku.

Načtěte `data/grouped-records.csv` do `b_data`. Obsahuje 36 zařízení po osmi měřeních. Do `b_features` vyberte pouze `temperature` a `vibration`, do `b_target` cílový `state` a do `b_groups` identifikátor `group_id`. ID není vstupem modelu. Data neobsahují chybějící hodnoty.

Prohlédněte počty řádků a zařízení. Měření stejného zařízení spolu souvisejí, proto při hodnocení nových zařízení patří vždy do stejné části dat.

### Ukázka: celé skupiny spolu

Čtyři skupiny mají po dvou řádcích. Parametr `test_size` se vztahuje k počtu skupin; všechny řádky jedné skupiny zůstanou spolu.

In [ ]:
demo_groups = np.repeat(["G1", "G2", "G3", "G4"], 2)  # Identifikátory čtyř malých skupin.
demo_features = np.arange(8).reshape(-1, 1)  # Osm řádků s jedním ilustračním vstupem.
demo_group_splitter = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=0)  # Jedno dělení celých skupin.
demo_group_train, demo_group_test = next(demo_group_splitter.split(demo_features, groups=demo_groups))  # Skupiny se předávají metodě split.
print(np.unique(demo_groups[demo_group_train]))  # Skupiny přidělené do tréninku.
print(np.unique(demo_groups[demo_group_test]))  # Odlišné skupiny přidělené do testu.

### B1 · Skupinové dělení

Vytvořte `GroupShuffleSplit` s `n_splits=1`, `test_size=0.25` a `random_state=42`. Získejte pole pozic `b_train_idx` a `b_test_idx`. Do dělení skutečně předejte skupiny. Vypište počty zařízení a řádků v obou částech, zkontrolujte neprůnik zařízení a porovnejte podíly tříd.

<details><summary>Nápověda</summary>

Parametry patří konstruktoru. Metoda `.split` dostává vstupy, cíle a `groups`; vrací postupně dvojice indexů. První dvojici lze získat pomocí `next`. `test_size` zde vyjadřuje podíl zařízení. Pro výběr řádků podle pozic slouží `.iloc`, pro počet různých ID `.nunique()`.

</details>

### Ukázka: učení a předpověď třídy

Trénovací a testovací řádky jsou zde zadané zvlášť. Model dostane při předpovědi pouze vstupy, skutečné cíle se použijí až při výpočtu přesnosti.

In [ ]:
demo_class_inputs = np.array([[1], [2], [4], [5]])  # Čtyři trénovací vzorky s jedním vstupem.
demo_class_targets = np.array([0, 0, 1, 1])  # Trénovací třídy ve stejném pořadí.
demo_classifier = DecisionTreeClassifier(max_depth=1, random_state=0)  # Malý strom s pevným nastavením.
demo_classifier.fit(demo_class_inputs, demo_class_targets)  # Učení pouze z trénovacích řádků.
demo_class_prediction = demo_classifier.predict([[1.5], [4.5]])  # Dva nové vstupy bez předání cílových tříd.
print(demo_class_prediction, accuracy_score([0, 1], demo_class_prediction))  # Předpovězené třídy a jejich přesnost.

### B2 · Model a interpretace

Naučte `DecisionTreeClassifier(max_depth=3, random_state=42)` pouze na trénovacích zařízeních. Pro testovací řádky vytvořte předpovědi a spočítejte `accuracy_score`. Nastavení je předem dané a podle výsledku testu se nemění.

<details><summary>Nápověda</summary>

`.fit` dostává trénovací vstupy a cíle; `.predict` pouze testovací vstupy. `accuracy_score` porovnává skutečné testovací cíle s předpověďmi. Identifikátory zařízení slouží jen ke skupinovému dělení.

</details>

**Výstup:** počty zařízení, podíly tříd a přesnost.

**K zamyšlení:** Jakou jinou úlohu by hodnotilo náhodné dělení jednotlivých měření? Proč samotné skupinové dělení nemusí zachovat podíly tříd?

[Dokumentace skupinového dělení](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GroupShuffleSplit.html).

## C · Předpověď další hodiny

**Cíl:** porovnat rozšiřující se a posuvné trénovací okno, oba typy s mezerou jedné hodiny i bez mezery.

Načtěte `data/time-series.csv` do `c_data`; sloupce `timestamp` a `target_available_time` převeďte při načtení na časové údaje pomocí `parse_dates`. Ověřte jedinečnost a chronologické pořadí časů. Soubor má pravidelný hodinový krok.

V čase `timestamp` jsou známé vstupy `current_value` a `lag1`. Cíl `next_value` se stane známým v `target_available_time`, o hodinu později. Při rovnosti s okamžikem učení je právě doručené měření již dostupné.

Prvních 80 % řádků uložte do `c_development`. Zbylých 20 % ponechte v `c_holdout` jako konečný test, který se v této úloze nevyhodnocuje. Z vývojové části vytvořte `c_features` se dvěma vstupy a `c_target` s cílem. V pracovní kopii vstupů pro tento experiment nahraďte každou třináctou hodnotu `lag1` chybějící hodnotou, počínaje první; samotný CSV soubor zůstane beze změny.

<details><summary>Nápověda</summary>

Hranici lze vypočítat z počtu řádků; souvislé části vybírá `.iloc`. Pro nezávislou pracovní kopii tabulky slouží `.copy()`. Chybějící číselnou hodnotu označuje `numpy.nan`. Pořadí času ověří `.is_monotonic_increasing`, jedinečnost `.is_unique`.

</details>

### Ukázka: mezera mezi časovými okny

Dvanáct hodinových řádků stačí pro dvě malá validační okna. Ukázka omezuje trénink na čtyři řádky. Bez `max_train_size` by se historie rozšiřovala.

In [ ]:
demo_times = pd.date_range("2025-02-01", periods=12, freq="h")  # Časy samostatné ukázkové řady.
for demo_gap in [0, 1]:  # Porovnání bez mezery a s jedním vynechaným řádkem.
    demo_time_splitter = TimeSeriesSplit(n_splits=2, test_size=2, gap=demo_gap, max_train_size=4)  # Dvě validační okna po dvou hodinách.
    for demo_time_train, demo_time_valid in demo_time_splitter.split(demo_times):  # Indexy každého časového kola.
        print(demo_gap, demo_time_train, demo_time_valid)  # Viditelná mezera a délka trénovací historie.
        print(demo_times[demo_time_train[-1]], demo_times[demo_time_valid[0]])  # Poslední čas tréninku a první čas validace.

### C1 · Časová okna

Vytvořte čtyři kombinace: rozšiřující se a posuvné okno, každé pro `gap=1` a `gap=0`. Použijte `TimeSeriesSplit` s `n_splits=4` a `test_size=24`. Posuvné okno navíc omezuje trénink na nejvýše 72 řádků pomocí `max_train_size`.

Pro každou kombinaci získejte trénovací a validační indexy výhradně v `c_features`. Zobrazte počty řádků a časy v jednotlivých kolech. Ověřte, že trénink předchází validaci a že konečný test zůstal mimo experiment.

<details><summary>Nápověda</summary>

`.split` vrací dvojice indexů, které lze postupně procházet smyčkou nebo převést na seznam. Bez omezení `max_train_size` zůstává všechna historie. `gap` vynechává řádky mezi tréninkem a validací; sám nenahrazuje kontrolu dostupnosti cílů ani nezaručuje nezávislost.

</details>

### Ukázka: kdy už je cíl známý

Tři kandidátní cíle přicházejí v různých časech. Rovnost s okamžikem učení zde znamená, že právě doručený cíl už lze použít.

In [ ]:
demo_available = pd.to_datetime(["2025-02-01 09:00", "2025-02-01 10:00", "2025-02-01 11:00"])  # Časy, kdy se kandidátní cíle stávají známými.
demo_fit_time = pd.Timestamp("2025-02-01 10:00")  # Předem daný okamžik učení.
demo_known = demo_available <= demo_fit_time  # Maska dostupných cílů připouští rovnost.
print(demo_known, demo_available[demo_known])  # Poslední cíl ještě není dostupný.

### Ukázka: pipeline a chyba předpovědi

Náhrada chybějící hodnoty se učí společně s modelem jen z malé trénovací sady. MAE (mean absolute error, průměrná absolutní chyba) porovnává předpovědi s výsledky oddělených validačních řádků.

In [ ]:
demo_reg_inputs = np.array([[1.0], [np.nan], [3.0], [4.0]])  # Čtyři trénovací řádky s jednou chybějící hodnotou.
demo_reg_targets = np.array([2.0, 3.0, 4.0, 5.0])  # Známé trénovací cíle.
demo_reg_model = Pipeline([("impute", SimpleImputer(strategy="median")), ("model", Ridge(alpha=0.5))])  # Náhrada hodnot a model tvoří jeden postup.
demo_reg_model.fit(demo_reg_inputs, demo_reg_targets)  # Náhrada i model se naučí pouze z těchto řádků.
demo_reg_prediction = demo_reg_model.predict([[2.0], [5.0]])  # Předpovědi pro dva oddělené validační vstupy.
demo_reg_observed = np.array([3.0, 6.0])  # Skutečné validační výsledky pro hodnocení.
print(demo_reg_prediction)  # Vypočtené číselné předpovědi.
print(mean_absolute_error(demo_reg_observed, demo_reg_prediction))  # Průměrná absolutní chyba ve stejných jednotkách jako cíl.

### C2 · Dostupné cíle a model

Pro každé kolo je `c_fit_time` první `timestamp` validačního okna. Z kandidátních trénovacích indexů vyberte jen ty, jejichž `target_available_time` nepřesahuje `c_fit_time`. Zobrazte počet zachovaných řádků a čas posledního dostupného cíle.

V každém kole vytvořte novou `Pipeline` s `SimpleImputer(strategy="median")` a `Ridge(alpha=1.0)`. Celou pipeline naučte pouze na povolených řádcích. Předpovězte validační vstupy. Jako srovnání použijte naivní předpověď: následující hodnota bude stejná jako právě známá `current_value`. Pro obě metody spočítejte průměrnou absolutní chybu (mean absolute error, MAE) pomocí `mean_absolute_error`.

Model se učí na začátku každého validačního bloku a uvnitř bloku se již nemění. Každá další hodinová předpověď využívá právě dostupné měření. Nejde o předpověď celého dne z jediného počátku. Předzpracování nesmí využívat budoucí hodnoty.

<details><summary>Nápověda: dostupnost</summary>

Časy dostupnosti porovnávejte pouze u kandidátních trénovacích řádků. Booleovská maska má mít stejnou délku jako pole jejich indexů; `.to_numpy()` ji převede na pole. Uložený čas trénovacího řádku nestačí, protože jeho cíl vzniká později.

</details>

<details><summary>Nápověda: pipeline a metriky</summary>

Pipeline dostává pojmenované kroky. `.fit` přijímá trénovací vstupy a cíle, `.predict` validační vstupy. Medián se tak odhadne uvnitř příslušného tréninku. MAE porovnává skutečné validační cíle s předpověďmi ve stejném pořadí. `c_holdout` nepatří do žádného výpočtu chyby ani výběru postupu.

</details>

### Výstup úlohy C

Vytvořte tabulku všech 16 validačních kol.

**K zamyšlení:**

- Ve kterých kolech model překonal naivní předpověď? Mění se závěr s oknem či mezerou?
- Proč se medián učí znovu v každém kole?
- Splnila i nulová mezera kontrolu dostupnosti cílů? Co znamená rovnost času dostupnosti a času učení?
- Co by bylo jinak při předpovědi celého dne z jediného počátečního měření?

Konečný test zůstává odložený. Výsledky této úlohy jsou validační, nikoli konečné testové skóre.

[Časová okna a mezera](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.TimeSeriesSplit.html), [pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html), [doplnění hodnot](https://scikit-learn.org/stable/modules/generated/sklearn.impute.SimpleImputer.html).